In [ ]:
####################################################################
# Question 1: Intro to the n-body problem
# To solve: closest distance, predict good eras in 3-body problem
####################################################################

class Body:
    def __init__(self,m,x,y,vx=0,vy=0):
        self.m = m     # mass
        self.x = x     # position
        self.y = y     # position
        self.vx = vx   # velocity
        self.vy = vy   # velocity

    def squareDist(self,other):
        return (self.x-other.x)**2+(self.y-other.y)**2

    def __str__(self):
        return f"Body({self.m},{self.x},{self.y},{self.vx},{self.vy})"

    def __repr__(self):
        return str(self)

    def asTuple(self):
        return (self.m,self.x,self.y,self.vx,self.vy)

        # Gives the next position and velocity of the current Body to its position
    # and velocity after time dt las elapsed, taking into account the pull
    # forces from the bodies in the array Bodies.
    def next(self, Bodies, dt):
        ret = Body(self.m, self.x, self.y)
        ax = ay = 0
        # for each p in Bodies we compute their force on this Body and add it to its acceleration
        for p in Bodies:
            if p == self: continue # current Body does not affect itself
            # euclidian distance between p and this Body
            sq_distance = self.squareDist(p)
            # see e.g. https://en.wikipedia.org/wiki/Newton%27s_law_of_universal_gravitation
            # for the vector form of Newton's law of gravity
            ax += (p.x - self.x) *  p.m * G / (sq_distance**1.5)
            ay += (p.y - self.y) *  p.m * G / (sq_distance**1.5)
        # we compute displacement due to acceleration that we computed above
        # and due to its current speed because of inertia
        ret.x += dt*dt*ax + dt*self.vx
        ret.y += dt*dt*ay + dt*self.vy
        # compute the velocity vectors
        ret.vx = (ret.x-self.x)/dt
        ret.vy = (ret.y-self.y)/dt

        return ret

    # Predict years of stability that self planet is going to have in a 3-sun solar system.
    # It suffices to take into account just the three suns (sunA,sunB,sunC) and the planet.
    # Stability is broken if one of the following criteria is violated:
    # a. No scorching: lA/dA + lB/dB + lC/dC must be less than scoeff
    # b. No freezing: lA/dA + lB/dB + lC/dC must be greater than fcoeff
    # Notes:
    # - dA, dB, dC are the squared distances between sunA, sunB, sunC and self, and
    # - lA, lB, lC are the luminosities of sunA, sunB, sunC
    # respectively.
    def threeBodyProblem(self,sunA,sunB,sunC,lA,lB,lC): # 15%
        duration = 0
        sim = Simulation([sunA, sunB, sunC, self])
        pss = sim.run()
        for p in pss:
            # calculating the total luminosities
            # relative luminosity of A, B and C (based on their distances from the planet)
            luminosityA = lA / p[3].squareDist(p[0])
            luminosityB = lB / p[3].squareDist(p[1])
            luminosityC = lC / p[3].squareDist(p[2])
            totalLuminosity = luminosityA + luminosityB + luminosityC
            # print(totalLuminosity) # test code

            if totalLuminosity > scoeff or totalLuminosity < fcoeff:
                break

            duration += sim.dt

        return duration


class Simulation:
    # Default simulation time 10yr, step time 0.01yr
    def __init__(self, Bodies, total_time = 10, dt=0.01):
        self.bodies = Bodies
        self.total_time = total_time
        self.dt = dt
        self.timesteps = int(total_time/dt)

    # Runs the simulation and produces an array of arrays of Bodies.
    # The t-th entry in the list is the position of Bodies after the t-th timestep
    # has been simulated
    def run(self):
        pss = [None]*(self.timesteps+1)
        pss[0] = self.bodies
        for t in range(self.timesteps):
            # for every Body in the current timestep add its next position in next timestep
            pss[t+1] = [pss[t][i].next(pss[t],self.dt) for i in range(len(self.bodies))]
        return pss

    # ADDED METHOD = CLOSEST DISTANCE O(n^3) TIME COMPLEXITY
    def closestDistance(self): # 15%
        if len(self.bodies) < 2:
            return None

        pss = self.run() # array of arrays of bodies for each timestep

        minDistance = pss[0][0].squareDist(pss[0][1])
        # loop through each array of bodies in the array
        for p in pss:
            # find the minimum distance between bodies in any of the arrays
            for b1 in p:
                for b2 in p:
                    if b1 == b2:
                        continue
                    distance = b1.squareDist(b2)
                    if distance < minDistance:
                        minDistance = distance

        return math.sqrt(minDistance)

    # Shows an animation of the simulation using matplotlib
    def show(self,x0,y0,x1,y1):
        pss = self.run()
        # get figure and axes objects
        fig, ax = plt.subplots()
        # set some reasonable zoom on axes
        ax.set_xlim(x0,x1)
        ax.set_ylim(y0,y1)
        # put labels on the window for Bodies with different colours
        scatter = []
        for i in range(len(self.bodies)):
            scatter.append(ax.scatter([], [], marker='o', label=f'Body {i}'))
        # add timestep text to the legend
        time_text = ax.text(0.02, 0.98, '', transform=ax.transAxes,
                            verticalalignment='top', bbox=dict(boxstyle='round',
                                                               facecolor='wheat', alpha=0.5))
        # update function from one position to the next
        def update(frame):
            for i in range(len(self.bodies)):
                scatter[i].set_offsets([pss[frame][i].x, pss[frame][i].y])
            time_text.set_text(f'Timestep: {frame}')
            return scatter + [time_text]
        # generate and show the animation
        a = FuncAnimation(fig, update, frames=len(pss), interval=1, blit=True, repeat=False)
        plt.xlabel("X coordinate (AU)")
        plt.ylabel("Y coordinate (AU)")
        plt.title("Celestial body trajectories")
        plt.legend()
        plt.show()

####################################################################
# Question 2: The tree Gadget
# To solve: add and remove elements on a Gadget
####################################################################

class Gadget:
    def __init__(self,box):
        self.root = GNode(box)
        self.size = 0

    def addToInternalNode (self,interalNode,p):
        isLeafNotFound = True
        currentInteralNode = interalNode

        while isLeafNotFound:

            for i in range(len(currentInteralNode.children)):
                # checks if the node added is within the current child
                if not currentInteralNode.children[i].box.isIn(p):
                    continue
                    # checks if the node is a interal node
                if not currentInteralNode.children[i].isLeaf():
                    currentInteralNode.nbodies +=1
                    currentInteralNode.updateCOM()
                    currentInteralNode = currentInteralNode.children[i]
                    continue

                currentInteralNode.nbodies +=1
                isLeafNotFound = False
                leafNode = currentInteralNode.children[i]


                if leafNode.nbodies == 0:
                    leafNode.p = p
                    leafNode.nbodies = 1
                    leafNode.updateCOM()
                    currentInteralNode.updateCOM()
                else:
                    boxes = leafNode.box.split4()

                    leafNode.children = [GNode(boxes[0]),GNode(boxes[1]),GNode(boxes[2]),GNode(boxes[3])]
                    leafNode.nbodies = 0
                    self.addToInternalNode(leafNode,p)
                    self.addToInternalNode(leafNode,leafNode.p)
                    leafNode.p = None
                    leafNode.updateCOM()
                    currentInteralNode.updateCOM()
                break
        interalNode.updateCOM()
    # Add Body p to this Gadget [30% TODO]
    def add(self,p):

        if self.root.isLeaf():
            if self.root.p is None:
                self.root.p = p
                self.root.nbodies =1
                self.root.updateCOM()
                self.size += 1
                return

            boxes = self.root.box.split4()
            self.root.nbodies = 0
            self.root.children = [GNode(boxes[0]),GNode(boxes[1]),GNode(boxes[2]),GNode(boxes[3])]
            currentBody = self.root.p
            self.addToInternalNode(self.root,p)
            self.addToInternalNode(self.root,currentBody)
            self.root.p = None
            self.root.updateCOM()
            self.size += 1
            return


        self.addToInternalNode(self.root,p)
        self.root.updateCOM()
        self.size += 1

    def removeRec (self, node,p):
        if node.isLeaf():
            if node.p is None: return False
            if node.p == p:
                node.p = None
                node.nbodies = 0
                node.updateCOM()
                return True
            return False

        hasRemoved = False
        for i in node.children:
            hasRemoved =  self.removeRec(i,p) or hasRemoved

        if not hasRemoved:
            return False

        node.nbodies -=1

        if node.nbodies < 2:
            for i in node.children:
                if i.p is not None:
                    node.p = i.p
            node.children = None
        node.updateCOM()
        return True


    # Remove Body p from this Gadget [20% TODO]
    def remove(self,p):
        self.removeRec(self.root,p)
        self.size -= 1


    def __str__(self):
        return self.root.niceStr()

    # Collect all Bodies from this Gadget and return them in an array
    def getBodies(self):
        A = [None]*self.size
        i = 0; stack = Stack(); stack.push(self.root)
        while not stack.isEmpty():
            n = stack.pop()
            if n.isLeaf():
                if n.p is not None:
                    A[i] = n.p
                    i += 1
            else:
                for c in n.children: stack.push(c)
        return A

        # Build a new Gadget and add in it all Bodies from array ps
    @staticmethod
    def fromBodies(ps):
        # calculate bottom-left and top-right positions
        if ps == []: return None
        x0 = x1 = ps[0].x
        y0 = y1 = ps[0].y
        for p in ps:
            if p.x < x0: x0 = p.x
            elif p.x > x1: x1 = p.x
            if p.y < y0: y0 = p.y
            elif p.y > y1: y1 = p.y
        # build gadget and add bodies
        g = Gadget(Box(x0,y0,x1,y1))
        for p in ps: g.add(p)
        return g

    @staticmethod
    def _drawNode(ax, node, showCom):
        x0, y0, x1, y1, mx, my = node.box.asTuple()
        w = x1-x0; h = y1-y0
        # draw the node rectangle
        rect = Rectangle((x0, y0), w, h, fill=False, linewidth=0.8, alpha=0.5)
        ax.add_patch(rect)
        # draw COM for internal nodes
        if showCom and not node.isLeaf():
            ax.plot([node.COM.x], [node.COM.y], marker='+', markersize=6)
            ax.plot([mx, node.COM.x], [my, node.COM.y], linewidth=0.3)
            # draw leaf contents
        if node.isLeaf():
            if node.p is not None:
                ax.plot([node.p.x], [node.p.y], marker='o', markersize=3)
        else:
            # recurse into children [NE, NW, SW, SE]
            for c in node.children:
                if c is not None:
                    Gadget._drawNode(ax, c, showCom)

    def plot(self, figsize=(6,6)): # figsize : (w,h) in inches
        showCom=True         # Draw line with '+' from center to COM of internal nodes
        margin_ratio=0.05    # Extra margin around the root box
        fig, ax = plt.subplots(figsize=figsize)
        # Plot node bounds
        Gadget._drawNode(ax, self.root, showCom)
        # collect Bodies from leaves
        A = self.getBodies()
        if A != []:
            xs, ys = zip(*[(A[i].x,A[i].y) for i in range(len(A))])
            ax.plot(xs, ys, linestyle='none', marker='o', markersize=3)
        # Set view limits from root box (with margin)
        x0, y0, x1, y1, mx, my = self.root.box.asTuple()
        dx, dy = x1-x0, y1-y0
        pad_x = margin_ratio * max(dx, 1e-12)
        pad_y = margin_ratio * max(dy, 1e-12)
        ax.set_xlim(x0-pad_x, x1+pad_x)
        ax.set_ylim(y0-pad_y, y1+pad_y)
        ax.set_aspect('equal', adjustable='box')
        plt.tight_layout()
        plt.show()

####################################################################
# Question 3: Using the Gadget to speed up simulations
# To solve: get bodies from a Gadget wrt reference body
####################################################################
class FastSimulation(Simulation):
    # Runs the simulation and produces an array of arrays of Bodies.
    # The t-th entry in the list is the position of Bodies after the t-th timestep
    # has been simulated.
    # Simulation uses a Gadget in order to approximate for each body the list
    # of other bodies gravitationally affecting its trajectory.
    def run(self,test=None):
        pss = [None]*(self.timesteps+1)
        pss[0] = self.bodies
        for t in range(self.timesteps):
            # calculate the current Gadget
            g = Gadget.fromBodies(pss[t])
            # for every Body in the current timestep add its next position in next timestep
            # but using the gadget g
            A = pss[t][:]
            for i in range(len(A)):
                new_ps = FastSimulation.getBodies(g,A[i])
                A[i] = A[i].next(new_ps,self.dt)
                if test is not None: test[i] = new_ps
            pss[t+1] = A
        return pss

    # Barnes-Hut criterion deciding whether node n should be opened when calculating
    # the gravitational forces exerted on body p.
    # By defualt, if p is in n then we open.
    @staticmethod
    def BarnesHut(n,p):
        theta = 0.7
        if n.box.isIn(p): return True
        sqDist = (n.COM.x-p.x)**2+(n.COM.y-p.y)**2
        return n.box.maxSide**2/sqDist >= theta**2

    # Get bodies from gadget g in an optimal way, using criterion shouldOpen and body p to
    # decide whether nodes in the gadget should be "opened" (i.e. their subnodes examined)
    # or not (i.e. the whole subtree approximated by its COM).
    @staticmethod
    def getBodies(g,p,shouldOpen=BarnesHut): # 20%

        new_ps = Stack()
        newStack = Stack()
        newStack.push(g.root)

        while not newStack.isEmpty():

            node = newStack.pop()
            if node.isLeaf():
                if node.p is not None:
                    new_ps.push(node.p)
                else:
                    pass
            else:
                if shouldOpen(node,p):
                    for child in node.children:
                        if child is not None:
                            newStack.push(child)
                else:
                    new_ps.push(node.COM)
        return new_ps.toArray()    